# Customer Segmentation Classification
IF3270 Machine Learning — Practicum 1

Steven Owen - 13523103
Daniel Pedrosa Wu - 13523099

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.base import clone

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load Data

In [ ]:
TARGET      = "Segmentation"
ID_COL      = "ID"
TRAIN_PATH  = "Train_processed.csv"
TEST_PATH   = "test_processed_no_solution.csv"
SCORING     = "f1_macro"
F1_AVERAGE  = "macro"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

df_train.head()

---
## 3. Exploratory Data Analysis
### 3.1 Data Integrity

In [ ]:
print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
print(f"\nDuplicate rows (train): {df_train.duplicated().sum()}")
print(f"Duplicate rows (test):  {df_test.duplicated().sum()}")

df_train.info()

In [ ]:
missing = df_train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing.plot.barh(ax=ax, color="salmon")
    ax.set_title("Missing Values per Feature")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values.")

### 3.2 Feature Separation (Dynamic)

In [ ]:
feature_cols = [c for c in df_train.columns if c not in [ID_COL, TARGET]]

num_cols = df_train[feature_cols].select_dtypes(include="number").columns.tolist()
cat_cols = df_train[feature_cols].select_dtypes(exclude="number").columns.tolist()

print(f"Numerical  ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")

### 3.3 Univariate Analysis

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    sns.histplot(df_train[col], kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    sns.boxplot(x=df_train[col], ax=axes[1])
    axes[1].set_title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in cat_cols:
    fig, ax = plt.subplots(figsize=(8, 3))
    order = df_train[col].value_counts().index
    sns.countplot(data=df_train, x=col, order=order, ax=ax)
    ax.set_title(f"Count of {col}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
order = df_train[TARGET].value_counts().index
sns.countplot(data=df_train, x=TARGET, order=order, ax=ax, palette="Set2")
ax.set_title(f"Target Distribution — {TARGET}")
plt.tight_layout()
plt.show()

df_train[TARGET].value_counts(normalize=True).round(4)

### 3.4 Multivariate Analysis

In [ ]:
if len(num_cols) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        df_train[num_cols].corr(),
        annot=True, fmt=".2f", cmap="coolwarm", ax=ax, vmin=-1, vmax=1
    )
    ax.set_title("Correlation Heatmap (Numerical Features)")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in num_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df_train, x=TARGET, y=col, ax=ax, palette="Set2")
    ax.set_title(f"{col} vs {TARGET}")
    plt.tight_layout()
    plt.show()

for col in cat_cols:
    ct = pd.crosstab(df_train[col], df_train[TARGET], normalize="index")
    ct.plot.bar(stacked=True, figsize=(8, 4), colormap="Set2")
    plt.title(f"{col} vs {TARGET} (Proportion)")
    plt.ylabel("Proportion")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title=TARGET, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

### Question 1

*Your answer here…*

### Question 2

*Your answer here…*

### Question 3

*Your answer here…*

---
## 4. Data Preprocessing

In [ ]:
COLUMNS_TO_DROP = []
df_train.drop(columns=COLUMNS_TO_DROP, inplace=True)
df_test.drop(columns=COLUMNS_TO_DROP, inplace=True)

num_cols = [c for c in num_cols if c not in COLUMNS_TO_DROP]
cat_cols = [c for c in cat_cols if c not in COLUMNS_TO_DROP]
feature_cols = [c for c in feature_cols if c not in COLUMNS_TO_DROP]

In [ ]:
def remove_outliers_iqr(data, cols):
    mask = pd.Series(True, index=data.index)
    for col in cols:
        Q1  = data[col].quantile(0.25)
        Q3  = data[col].quantile(0.75)
        IQR = Q3 - Q1
        mask &= data[col].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return data[mask]

before = df_train.shape[0]
df_train = remove_outliers_iqr(df_train, num_cols)
print(f"Rows removed: {before - df_train.shape[0]}  ({before} → {df_train.shape[0]})")

In [ ]:
df = df_train.copy()

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.isnull().sum().sum()

In [ ]:
label_enc = LabelEncoder()
df[TARGET] = label_enc.fit_transform(df[TARGET])

ordinal_cols = [c for c in cat_cols if df[c].nunique() <= 5]
onehot_cols  = [c for c in cat_cols if df[c].nunique() > 5]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ordinal_cols),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), onehot_cols),
    ],
    remainder="drop",
)

print(f"Ordinal-encoded: {ordinal_cols}")
print(f"One-Hot-encoded:  {onehot_cols}")

In [ ]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}")
print(f"Class distribution (train): {np.bincount(y_train)}")

### Preprocessing Rationale

*Explain your preprocessing decisions here…*

---
## 5. Modeling — Ensemble Learning

We evaluate four ensemble strategies: **Bagging**, **Boosting**, **Voting**, and **Stacking**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

### 5.1 Bagging — Random Forest

In [ ]:
pipe_rf = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring=SCORING)
results["Random Forest"] = scores_rf.mean()

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_val)

print(f"CV {SCORING}: {scores_rf.mean():.4f} \u00b1 {scores_rf.std():.4f}")

### 5.2 Boosting — Gradient Boosting

In [ ]:
pipe_gb = Pipeline([
    ("pre", preprocessor),
    ("clf", GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)),
])

scores_gb = cross_val_score(pipe_gb, X_train, y_train, cv=cv, scoring=SCORING)
results["Gradient Boosting"] = scores_gb.mean()

pipe_gb.fit(X_train, y_train)
y_pred_gb = pipe_gb.predict(X_val)

print(f"CV {SCORING}: {scores_gb.mean():.4f} \u00b1 {scores_gb.std():.4f}")

### 5.3 Voting — Heterogeneous Ensemble

In [ ]:
estimators_vote = [
    ("rf",  Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])),
    ("gb",  Pipeline([("pre", preprocessor), ("clf", GradientBoostingClassifier(n_estimators=200, random_state=42))])),
    ("dt",  Pipeline([("pre", preprocessor), ("clf", DecisionTreeClassifier(max_depth=8, random_state=42))])),
]

pipe_vote = VotingClassifier(
    estimators=estimators_vote,
    voting="hard",
    weights=[2, 2, 1],
)

scores_vote = cross_val_score(pipe_vote, X_train, y_train, cv=cv, scoring=SCORING)
results["Voting"] = scores_vote.mean()

pipe_vote.fit(X_train, y_train)
y_pred_vote = pipe_vote.predict(X_val)

print(f"CV {SCORING}: {scores_vote.mean():.4f} \u00b1 {scores_vote.std():.4f}")

### 5.4 Stacking — Meta-Learner

In [ ]:
estimators_stack = [
    ("rf",  Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])),
    ("gb",  Pipeline([("pre", preprocessor), ("clf", GradientBoostingClassifier(n_estimators=200, random_state=42))])),
    ("dt",  Pipeline([("pre", preprocessor), ("clf", DecisionTreeClassifier(max_depth=8, random_state=42))])),
]

pipe_stack = StackingClassifier(
    estimators=estimators_stack,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=cv,
    n_jobs=-1,
)

scores_stack = cross_val_score(pipe_stack, X_train, y_train, cv=cv, scoring=SCORING)
results["Stacking"] = scores_stack.mean()

pipe_stack.fit(X_train, y_train)
y_pred_stack = pipe_stack.predict(X_val)

print(f"CV {SCORING}: {scores_stack.mean():.4f} \u00b1 {scores_stack.std():.4f}")

---
## 6. Evaluation — Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": results.keys(),
    "CV Macro-F1": [round(v, 4) for v in results.values()],
}).sort_values("CV Macro-F1", ascending=False).reset_index(drop=True)

comparison.style.highlight_max(subset=["CV Macro-F1"], color="lightgreen")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=comparison, x="CV Macro-F1", y="Model", palette="viridis", ax=ax)
ax.set_title("Model Comparison — CV Macro-F1")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---
## 7. Error Analysis

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_models = {
    "Random Forest": (pipe_rf, y_pred_rf),
    "Gradient Boosting": (pipe_gb, y_pred_gb),
    "Voting": (pipe_vote, y_pred_vote),
    "Stacking": (pipe_stack, y_pred_stack),
}
best_pipe, y_pred_best = best_models[best_name]

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_val, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=label_enc.classes_).plot(ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

print(classification_report(y_val, y_pred_best, target_names=label_enc.classes_))

In [ ]:
mask = y_val != y_pred_best
misclassified = X_val[mask].copy()
misclassified["Actual"]    = label_enc.inverse_transform(y_val[mask])
misclassified["Predicted"] = label_enc.inverse_transform(y_pred_best[mask])

print(f"Misclassified samples: {misclassified.shape[0]} / {X_val.shape[0]}")
misclassified.head(10)

---
## 8. Kaggle Submission

In [ ]:
pipe_final = clone(best_pipe)

X_full = df[feature_cols]
y_full = df[TARGET]
pipe_final.fit(X_full, y_full)

df_test_clean = df_test.copy()
for col in num_cols:
    if col in df_test_clean.columns:
        df_test_clean[col].fillna(df[col].median(), inplace=True)
for col in cat_cols:
    if col in df_test_clean.columns:
        df_test_clean[col].fillna(df[col].mode()[0], inplace=True)

test_preds = pipe_final.predict(df_test_clean[feature_cols])
test_labels = label_enc.inverse_transform(test_preds)

submission = pd.DataFrame({
    ID_COL: df_test[ID_COL],
    TARGET: test_labels,
})

submission.to_csv("submission.csv", index=False)
print(f"Submission saved — {submission.shape[0]} rows")
submission.head()